# Fine-tune MedImageParse3D on your own data

End-to-end: convert your own volumes into the slice format the trainer expects, sanity-check
them, then launch a fine-tuning job with the **registered** `medimageparse_3d_finetune`
component from the `azureml` registry.

Fine-tuning runs the **2D** BiomedParse model over pre-sliced **2.5D RGB PNGs**, so volumes
have to be converted offline. [`prepare_finetune_data.py`](prepare_finetune_data.py) does this
and reproduces the exact preprocessing used at inference time (slice-axis choice, square pad,
512 resize, 0-255 scaling, and `ch0 = slice d` / `ch1 = d-1` / `ch2 = d+1` packing), so
fine-tuning does not introduce a train/serve skew. See [README.md](README.md) for details.

### Prerequisites

- Paired image/label volumes (NIfTI, MHA or NRRD). Labels must be an **integer class map**:
  `0` = background, `1..N` = your targets.
- The `healthcareai_toolkit` package installed (see the repo root README), or at minimum
  `nibabel`, `SimpleITK`, `opencv-python` and `azure-ai-ml` in the active kernel. The
  converter reads NIfTI with `nibabel` and MHA/NRRD with `SimpleITK`.
- A `config.json` in this folder pointing at your workspace.
- `az login` (or a managed identity) with access to that workspace.


## 1. Configure

Everything you need to change lives in this cell.

In [ ]:
from pathlib import Path

# --- your raw data ----------------------------------------------------------
RAW_IMAGES = "raw/images"        # folder of image volumes
RAW_LABELS = "raw/labels"        # folder of label volumes (integer class maps)
LABEL_SUFFIX = "_seg"            # labels are <stem><LABEL_SUFFIX>.nii.gz; "" if identical

# Class id -> target name. Ids must match the pixel values in your label volumes.
CLASSES = {
    "1": "spleen",
    "2": "right kidney",
    "3": "left kidney",
    "6": "liver",
    "10": "pancreas",
}
MODALITY = "CT"                  # used to phrase the text prompts, e.g. "MR", "CT", "PET"
SITE = "abdomen"                 # e.g. "brain", "chest"
CT_WINDOW = "40,400"             # CT level,width. Set to None for percentile scaling.
NEGATIVE_RATIO = 0.15            # empty slices kept, as a fraction of foreground slices

# --- output -----------------------------------------------------------------
DATASET_DIR = "MY_DATASET"       # local folder the converter writes
ASSET_NAME = "mip3d-own-data"    # Azure ML data asset name

# --- training hyperparameters ----------------------------------------------
MAX_EPOCHS = 10
TRAIN_BATCH_SIZE = 4
LEARNING_RATE = 2e-5
NUM_GPUS = 1                     # per node; must be <= the GPUs on COMPUTE_SIZE
NUM_NODES = 1

# --- Azure ML ---------------------------------------------------------------
COMPUTE_NAME = "hls-4xA100"
COMPUTE_SIZE = "Standard_NC96ads_A100_v4"   # only used if the cluster must be created
EXPERIMENT_NAME = "medimageparse_3d_finetune_experiment"

# On hosts exposing several managed identities, DefaultAzureCredential may pick
# one without workspace access. Set the client id that does have it, or None.
AZURE_CLIENT_ID = None

# Workspaces with shared-key auth disabled on workspaceblobstore cannot use the
# SDK's local-folder upload. Name an identity-based datastore to stage through
# instead, or set to None to use the workspace default.
UPLOAD_DATASTORE = "olympus_external"
UPLOAD_PREFIX = "mip3d_finetune"

print(f"{len(CLASSES)} classes -> {sorted(int(k) for k in CLASSES)}")

## 2. Convert your volumes to training slices

This writes `train/`, `train_mask/` and `train.json`, then validates the result. The validator
is not cosmetic: a mask exported as binary `{0, 255}` is **silently zeroed** by the dataset
(image *and* mask), so the job would train on nothing while still reporting success.

In [ ]:
import json
import subprocess
import sys

cmd = [
    sys.executable, "prepare_finetune_data.py",
    "--images-dir", RAW_IMAGES,
    "--labels-dir", RAW_LABELS,
    "--out", DATASET_DIR,
    "--classes", json.dumps(CLASSES),
    "--modality", MODALITY,
    "--site", SITE,
    "--negative-ratio", str(NEGATIVE_RATIO),
]
if LABEL_SUFFIX:
    cmd += ["--label-suffix", LABEL_SUFFIX]
if CT_WINDOW:
    cmd += ["--ct-window", CT_WINDOW]

print(" ".join(cmd), "\n")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError("data preparation failed - see the errors above")

### Visual check

Confirm the packing looks right *before* spending GPU hours. The first three panels are three
**adjacent slices**, not colour channels of a photo. In the composite you should see colour
fringing exactly where anatomy shifts between slices - that is the 3D context the 2D backbone
receives. The mask must be a class-id map, not a binary image.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

meta = json.loads(Path(DATASET_DIR, "train.json").read_text())
# Pick a slice with plenty of structure to look at.
best = max(
    meta["annotations"],
    key=lambda a: len(a["class_ids"]),
)
name = best["file_name"]

img = cv2.imread(str(Path(DATASET_DIR, "train", name)), cv2.IMREAD_UNCHANGED)
mask = cv2.imread(str(Path(DATASET_DIR, "train_mask", name)), cv2.IMREAD_UNCHANGED)

fig, ax = plt.subplots(1, 5, figsize=(21, 4.6))
ax[0].imshow(img[:, :, 0], cmap="gray"); ax[0].set_title("ch0 = slice d\n(segmented slice)")
ax[1].imshow(img[:, :, 1], cmap="gray"); ax[1].set_title("ch1 = slice d-1\n(3D context)")
ax[2].imshow(img[:, :, 2], cmap="gray"); ax[2].set_title("ch2 = slice d+1\n(3D context)")
ax[3].imshow(img[:, :, ::-1]);           ax[3].set_title("composite\n(fringing = 3D signal)")
vmax = max(int(k) for k in CLASSES)
m = ax[4].imshow(mask, cmap="nipy_spectral", vmin=0, vmax=vmax, interpolation="nearest")
present = sorted(int(v) for v in np.unique(mask) if v)
ax[4].set_title(f"mask (pixel == class id)\npresent: {present}")
plt.colorbar(m, ax=ax[4], fraction=0.046)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.suptitle(f"{name}   image {img.shape} {img.dtype}   mask {mask.shape} {mask.dtype}")
plt.tight_layout(); plt.show()

assert img.shape == (512, 512, 3) and mask.ndim == 2, "unexpected slice format"
print(f"{len(meta['annotations'])} slices ready in {DATASET_DIR}/")

## 3. Write the training hyperparameters

`parameters.yaml` is loaded last by Hydra, so it overrides everything in the component's
built-in config.

> **Multi-GPU:** `trainer.devices` alone is not enough - the component's
> `process_count_per_instance` must match, which is set on the pipeline node in step 6.

In [ ]:
params = f"""# MedImageParse3D fine-tuning overrides (merged last by Hydra)
datamodule:
  dataloaders:
    train:
      batch_size: {TRAIN_BATCH_SIZE}
    validate:
      batch_size: {TRAIN_BATCH_SIZE}

optimizer:
  lr: {LEARNING_RATE}
  weight_decay: 0.01

trainer:
  max_epochs: {MAX_EPOCHS}
  devices: {NUM_GPUS}
  num_nodes: {NUM_NODES}
"""

# The component's config has `- parameters` in its Hydra defaults list, and the
# launcher adds this file's folder to the search path, so the name must be exactly
# parameters.yaml. Kept in its own folder so it does not clobber the sample one.
params_dir = Path("own_data_config")
params_dir.mkdir(exist_ok=True)
params_path = params_dir / "parameters.yaml"
params_path.write_text(params)
print(params)

## 4. Connect to Azure ML

In [ ]:
import os

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities import AmlCompute, Data
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential

if AZURE_CLIENT_ID:
    os.environ["AZURE_CLIENT_ID"] = AZURE_CLIENT_ID

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential)
ml_registry = MLClient(credential=credential, registry_name="azureml")

# from_config only reads the local file, so make one real call to surface auth
# problems here rather than several cells later.
try:
    next(iter(ml_client.compute.list()), None)
except Exception as exc:
    raise RuntimeError(
        "Could not reach the workspace with the selected credential. If this host has "
        "multiple managed identities, set AZURE_CLIENT_ID in the config cell to the one "
        "granted access to the workspace."
    ) from exc

print("Workspace:", ml_client.workspace_name)

## 5. Upload the dataset

The mount root **is** the dataset root: the component reads `train/`, `train_mask/` and
`train.json` directly from the folder you register here.

In [ ]:
def upload_to_datastore(local_path: Path, datastore_name: str, remote_prefix: str) -> str:
    """Upload via AAD to a named datastore and return its azureml:// URI.

    Used when the workspace's default store has shared-key auth disabled, which
    breaks the SDK's built-in local-folder upload.
    """
    from azure.storage.blob import BlobServiceClient

    store = ml_client.datastores.get(datastore_name)
    service = BlobServiceClient(
        f"https://{store.account_name}.blob.core.windows.net", credential=credential
    )
    container = service.get_container_client(store.container_name)

    # Both a folder and a single file land under <prefix>/<name>.
    root = f"{remote_prefix}/{local_path.name}"
    files = (
        [local_path]
        if local_path.is_file()
        else sorted(p for p in local_path.rglob("*") if p.is_file())
    )
    for path in files:
        blob = root if local_path.is_file() else f"{root}/{path.relative_to(local_path).as_posix()}"
        with open(path, "rb") as handle:
            container.upload_blob(name=blob, data=handle, overwrite=True)
    print(f"  uploaded {len(files)} file(s) to {datastore_name}/{root}")

    return f"azureml://datastores/{datastore_name}/paths/{root}"


def register(local_path: Path, name: str, asset_type: str) -> Data:
    """Register a Data asset, staging through UPLOAD_DATASTORE when configured."""
    path = str(local_path)
    if UPLOAD_DATASTORE:
        path = upload_to_datastore(local_path, UPLOAD_DATASTORE, UPLOAD_PREFIX)
    asset = ml_client.data.create_or_update(Data(path=path, type=asset_type, name=name))
    print(f"  {asset.name} v{asset.version}")
    return asset


training_data = register(Path(DATASET_DIR), f"{ASSET_NAME}-training_data", AssetTypes.URI_FOLDER)
parameters = register(params_path, f"{ASSET_NAME}-parameters", AssetTypes.URI_FILE)

## 6. Compute

In [ ]:
try:
    compute = ml_client.compute.get(name=COMPUTE_NAME)
    print(f"Using existing compute {compute.name} ({compute.size})")
except ResourceNotFoundError:
    print(f"Creating {COMPUTE_NAME} ({COMPUTE_SIZE}) ...")
    compute = ml_client.compute.begin_create_or_update(
        AmlCompute(
            name=COMPUTE_NAME,
            description="GPU cluster for MedImageParse3D fine-tuning",
            min_instances=0,
            max_instances=NUM_NODES,
            size=COMPUTE_SIZE,
        )
    ).result()
    print("Created", compute.name)

## 7. Launch the fine-tuning job

Uses the registered `medimageparse_3d_finetune` component and the published
`MedImageParse3D` base model from the `azureml` registry.

In [ ]:
import random

finetune_component = ml_registry.components.get(
    name="medimageparse_3d_finetune", label="latest"
)
base_model = ml_registry.models.get(name="MedImageParse3D", label="latest")
print(f"Component {finetune_component.name} v{finetune_component.version}")
print(f"Base model {base_model.name} v{base_model.version}")


@pipeline(name=f"mip3d_finetune_own_data_{random.randint(0, 100000)}")
def finetune_pipeline():
    step = finetune_component(
        pretrained_mlflow_model=base_model.id,
        data=training_data.id,
        config=parameters.id,
    )
    # trainer.devices alone does not scale out; the launcher spawns one process
    # per GPU, so this has to match NUM_GPUS.
    step.distribution.process_count_per_instance = NUM_GPUS
    step.resources = {"instance_count": NUM_NODES}
    # "upload" writes outputs locally and pushes them via AAD at the end, which
    # works when the default store rejects key-based auth. Harmless otherwise.
    step.outputs.out.mode = "upload"
    step.outputs.mlflow_model_folder.mode = "upload"
    return {"mlflow_model_folder": step.outputs.mlflow_model_folder}


pipeline_object = finetune_pipeline()
pipeline_object.compute = compute.name
pipeline_object.settings.continue_on_step_failure = False
pipeline_object.outputs.mlflow_model_folder.mode = "upload"

pipeline_job = ml_client.jobs.create_or_update(
    pipeline_object, experiment_name=EXPERIMENT_NAME
)
pipeline_job_run_id = pipeline_job.name
print("\nSubmitted:", pipeline_job_run_id)
print("Studio:", pipeline_job.studio_url)

## 8. Monitor

Streams until the job finishes. Safe to interrupt - the job keeps running, and you can
reattach with `ml_client.jobs.get(pipeline_job_run_id)`.

In [ ]:
ml_client.jobs.stream(pipeline_job_run_id)
print("Final status:", ml_client.jobs.get(pipeline_job_run_id).status)

## 9. Register the fine-tuned model

The base model's `python_model.pkl` loads the vision weights by a fixed filename
(`vision_model_name = "biomedparse_3D_AllData_MultiView_edge.safetensors"`), but the fine-tune
component (<= 0.0.1) writes them as `boltzformer_focal_all.safetensors`. Left unchanged, the
online deployment fails at container start. The cell below renames the checkpoint in the job
output first - a fast server-side copy that also works when the store rejects key-based auth -
so the registered model is deployment-ready.


In [ ]:
from azure.ai.ml.entities import Model
from azure.storage.blob import BlobServiceClient, BlobSasPermissions, generate_blob_sas
import datetime as dt
import time

# The base model's python_model.pkl loads vision weights by this exact filename.
EXPECTED_CKPT = "biomedparse_3D_AllData_MultiView_edge.safetensors"

# The component writes the checkpoint under a different name, so rename it in the job
# output before registering. Server-side copy via an AAD user-delegation SAS, which
# also works on workspaces with key-based auth disabled.
child_run_id = next(iter(ml_client.jobs.list(parent_job_name=pipeline_job_run_id))).name
store = ml_client.datastores.get("workspaceblobstore")
service = BlobServiceClient(
    f"https://{store.account_name}.blob.core.windows.net", credential=credential
)
container = service.get_container_client(store.container_name)
ckpt_dir = f"azureml/{child_run_id}/mlflow_model_folder/artifacts/checkpoints"
checkpoints = [
    b.name
    for b in container.list_blobs(name_starts_with=f"{ckpt_dir}/")
    if b.name.endswith(".safetensors")
]

if not any(b.endswith(EXPECTED_CKPT) for b in checkpoints):
    source = checkpoints[0]
    now = dt.datetime.utcnow()
    udk = service.get_user_delegation_key(now, now + dt.timedelta(hours=1))
    sas = generate_blob_sas(
        account_name=store.account_name,
        container_name=store.container_name,
        blob_name=source,
        user_delegation_key=udk,
        permission=BlobSasPermissions(read=True),
        expiry=now + dt.timedelta(hours=1),
    )
    dest = container.get_blob_client(f"{ckpt_dir}/{EXPECTED_CKPT}")
    dest.start_copy_from_url(
        f"https://{store.account_name}.blob.core.windows.net/"
        f"{store.container_name}/{source}?{sas}"
    )
    while dest.get_blob_properties().copy.status == "pending":
        time.sleep(1)
    print(f"Renamed {source.split('/')[-1]} -> {EXPECTED_CKPT}")

run_model = ml_client.models.create_or_update(
    Model(
        path=f"azureml://jobs/{pipeline_job_run_id}/outputs/mlflow_model_folder",
        name=f"MIP3D-{ASSET_NAME}-{pipeline_job_run_id}",
        description=f"MedImageParse3D fine-tuned on {ASSET_NAME}",
        type=AssetTypes.MLFLOW_MODEL,
    )
)
print(f"Registered {run_model.name} v{run_model.version}")